# High-Value Possession Stories

A notebook for turning high-aEC UFA possessions into Shown Space-style story material: league-wide top possessions, team style comparisons, possession archetypes, and consistency checks.

## tl;dr

Run the notebook top to bottom to populate this section with the current 2026 league data. The first story angle is: highest `aec_per_throw` possessions and highest `total_aec` possessions are related, but they are not the same question.

- `aec_per_throw` finds the sharpest, most efficient points.
- `total_aec` finds the points that accumulated the most value over the whole possession.
- Shape features help turn those leaderboards into possession types: direct, winding, middle-heavy, sideline-heavy, reset-heavy, and side-switching.

## Context & Methods

This notebook uses Shown Space throw data as the source of truth for aEC. The main story pool is intentionally narrow: O-line scoring possessions, long-field starts, and no hucks. That filter is useful for studying possession shape because it removes obvious one-throw bombs and asks what high-value worked points look like.

### Key Assumptions

- Long-field means `start_y <= 45` and `field_progress >= 50`.
- Non-huck means zero throws with distance at least 40 yards.
- O-line and outcome labels come from the possession reconstruction helpers.
- Scoring-only views are not full team-quality rankings. Turnover-inclusive consistency views are included as a reality check.

## Metric Key

- `total_aec`: sum of aEC over all throws in the possession.
- `aec_per_throw`: `total_aec / throw_count`; the possession-level aEC/T rate.
- `team_aec_per_throw`: total team aEC divided by total team throws in the selected pool.
- `median_metric`: the middle possession for a team after sorting by the selected metric.
- `p75_metric`: the 75th percentile possession; roughly the team's good-but-not-only-best possession level.
- `top_n_mean_metric`: average of a larger top group, useful for consistency instead of one-point spikes.
- `high_metric_share`: share of possessions above the league-wide high-value threshold used in the consistency view.
- Shape columns: `shape_width`, `shape_directness`, `shape_side_switches`, `shape_middle_third_share`, `shape_sideline_share`, and `shape_backwards_share` describe how the point moved across the field.

## Setup

In [ ]:
import importlib
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import ufa_aec_possessions
import ufa_aec_possessions.browser
import ufa_aec_possessions.fetching
import ufa_aec_possessions.leaderboards
import ufa_aec_possessions.plotting
import ufa_aec_possessions.possessions
import ufa_aec_possessions.selection

importlib.reload(ufa_aec_possessions.browser)
importlib.reload(ufa_aec_possessions.fetching)
importlib.reload(ufa_aec_possessions.leaderboards)
importlib.reload(ufa_aec_possessions.plotting)
importlib.reload(ufa_aec_possessions.possessions)
importlib.reload(ufa_aec_possessions.selection)
importlib.reload(ufa_aec_possessions)

from ufa_aec_possessions import (
    add_possession_shape_features,
    build_possessions,
    build_scoring_possessions,
    compare_top_aec_metrics_by_team,
    create_team_aec_comparison_browser,
    fetch_shownspace_season_throws_cached,
    filter_analysis_possessions,
    plot_representative_paths,
    plot_side_by_side_paths,
    select_top_aec_possessions_league,
    summarize_team_aec_consistency,
    summarize_team_top_possessions,
)

In [ ]:
SEASON = 2026
MAX_GAMES = None
FORCE_REFRESH = False
CACHE_DIR = REPO_ROOT / 'data' / 'cache' / 'shownspace'

TOP_N = 5
CONSISTENCY_TOP_N = 10
CONSISTENCY_MIN_POSSESSIONS = 20
CONSISTENCY_MIN_THROWS = 5

DEFAULT_STORY_FILTER = dict(
    outcomes=('goal',),
    line_type='o_line',
    long_field_only=True,
    max_start_y=45,
    min_field_progress=50,
    exclude_hucks=True,
)

## Data

Load the cached Shown Space season data, then reconstruct both all possessions and scoring possessions. The all-possession version is used for turnover-inclusive checks; the scoring version is used for the possession-shape story.

In [ ]:
league_games, league_throws = fetch_shownspace_season_throws_cached(
    season=SEASON,
    max_games=MAX_GAMES,
    cache_dir=CACHE_DIR,
    force_refresh=FORCE_REFRESH,
)
league_all_possessions, league_all_paths = build_possessions(league_throws)
league_scoring_possessions, league_scoring_paths = build_scoring_possessions(league_throws)

print(f'League games loaded: {len(league_games):,}')
print(f'League throws loaded: {len(league_throws):,}')
print(f'All reconstructed possessions: {len(league_all_possessions):,}')
print(f'Scoring possessions: {len(league_scoring_possessions):,}')

## Story Helpers

Small presentation helpers for labeling possessions, formatting bounded tables, and assigning quick shape tags.

In [ ]:
def label_ranked_paths(possessions, paths, metric, prefix):
    lookup = {str(path['possession_id'].iloc[0]): path for path in paths if not path.empty}
    labeled = {}
    for rank, (_, row) in enumerate(possessions.reset_index(drop=True).iterrows(), start=1):
        possession_id = str(row['possession_id'])
        if possession_id not in lookup:
            continue
        team = str(row.get('team_id', '')).title()
        value = pd.to_numeric(pd.Series([row.get(metric)]), errors='coerce').iloc[0]
        label = f'{prefix} {rank}: {team} {value:.3f}' if pd.notna(value) else f'{prefix} {rank}: {team}'
        labeled[label] = lookup[possession_id]
    return labeled


def possession_story_table(possessions, metric, rank_column='league_rank'):
    columns = [
        rank_column, 'team_id', 'GameID', 'throw_count', 'total_aec', 'aec_per_throw',
        'start_y', 'field_progress', 'shape_width', 'shape_directness',
        'shape_middle_third_share', 'shape_sideline_share', 'shape_side_switches',
    ]
    table = possessions.reindex(columns=[column for column in columns if column in possessions]).copy()
    for column in ['total_aec', 'aec_per_throw', 'start_y', 'field_progress', 'shape_width', 'shape_directness', 'shape_middle_third_share', 'shape_sideline_share']:
        if column in table:
            table[column] = pd.to_numeric(table[column], errors='coerce').round(3)
    if 'team_id' in table:
        table['team_id'] = table['team_id'].astype(str).str.title()
    return table


def tag_possession_shape(row):
    tags = []
    if row.get('shape_directness', 0) >= 0.70:
        tags.append('direct')
    elif row.get('shape_directness', 1) <= 0.45:
        tags.append('winding')
    if row.get('shape_middle_third_share', 0) >= 0.55:
        tags.append('middle-heavy')
    if row.get('shape_sideline_share', 0) >= 0.25:
        tags.append('sideline-heavy')
    if row.get('shape_side_switches', 0) >= 3:
        tags.append('side-switching')
    if row.get('reset_count', 0) >= 2 or row.get('shape_backwards_share', 0) >= 0.25:
        tags.append('reset-heavy')
    if row.get('shape_large_gain_share', 0) >= 0.35:
        tags.append('chunk-gain')
    return ' | '.join(tags) if tags else 'balanced'


def add_shape_tags(frame):
    tagged = frame.copy()
    tagged['shape_tags'] = tagged.apply(tag_possession_shape, axis=1)
    return tagged

## Build The Story Pool

This is the main article-style pool: O-line, non-huck, long-field scoring possessions.

In [ ]:
story_possessions, story_paths = filter_analysis_possessions(
    league_scoring_possessions,
    league_scoring_paths,
    **DEFAULT_STORY_FILTER,
)
story_possessions = add_possession_shape_features(story_possessions, story_paths)
story_possessions = add_shape_tags(story_possessions)

print(f'Story pool possessions: {len(story_possessions):,}')
print(f'Teams represented: {story_possessions["team_id"].nunique():,}')
print(f'Median throws per possession: {story_possessions["throw_count"].median():.1f}')

## Results

### 1. League Top Five: aEC/T vs Total aEC

Use this as the first visual hook. The left field asks which points were most efficient per throw. The right field asks which points accumulated the most value across the entire possession.

In [ ]:
league_top_by_aec_t, league_top_paths_by_aec_t = select_top_aec_possessions_league(
    league_scoring_possessions,
    league_scoring_paths,
    metric='aec_per_throw',
    n=TOP_N,
    **DEFAULT_STORY_FILTER,
)
league_top_by_total_aec, league_top_paths_by_total_aec = select_top_aec_possessions_league(
    league_scoring_possessions,
    league_scoring_paths,
    metric='total_aec',
    n=TOP_N,
    **DEFAULT_STORY_FILTER,
)

league_top_by_aec_t = add_shape_tags(league_top_by_aec_t)
league_top_by_total_aec = add_shape_tags(league_top_by_total_aec)

plot_side_by_side_paths(
    label_ranked_paths(league_top_by_aec_t, league_top_paths_by_aec_t, 'aec_per_throw', 'aEC/T'),
    label_ranked_paths(league_top_by_total_aec, league_top_paths_by_total_aec, 'total_aec', 'Tot'),
    left_title='League top 5 by aEC per throw',
    right_title='League top 5 by total aEC',
    title=f'{SEASON} UFA top non-huck long-field scoring possessions',
)

In [ ]:
print('Top 5 by aEC/T')
display(possession_story_table(league_top_by_aec_t, 'aec_per_throw'))

print('Top 5 by total aEC')
display(possession_story_table(league_top_by_total_aec, 'total_aec'))

### 2. Team Style Browser

Flip through teams and compare their top five non-huck long-field scoring possessions by `aec_per_throw` and `total_aec`.

In [ ]:
team_metric_comparison = compare_top_aec_metrics_by_team(
    league_scoring_possessions,
    league_scoring_paths,
    metrics=('aec_per_throw', 'total_aec'),
    n=TOP_N,
    **DEFAULT_STORY_FILTER,
)
create_team_aec_comparison_browser(team_metric_comparison)

### 3. Possession Archetypes

This table turns the league top possessions into first-pass shape tags. The tags are deliberately simple; they are meant to suggest article sections, not be the final taxonomy.

In [ ]:
archetype_candidates = pd.concat(
    [
        league_top_by_aec_t.assign(selection='top aEC/T'),
        league_top_by_total_aec.assign(selection='top total aEC'),
    ],
    ignore_index=True,
).drop_duplicates('possession_id')

archetype_columns = [
    'selection', 'league_rank', 'team_id', 'shape_tags', 'throw_count',
    'total_aec', 'aec_per_throw', 'shape_width', 'shape_directness',
    'shape_middle_third_share', 'shape_sideline_share', 'shape_side_switches',
    'reset_count', 'GameID', 'possession_id',
]
archetype_table = archetype_candidates.reindex(columns=archetype_columns).copy()
for column in ['total_aec', 'aec_per_throw', 'shape_width', 'shape_directness', 'shape_middle_third_share', 'shape_sideline_share']:
    archetype_table[column] = pd.to_numeric(archetype_table[column], errors='coerce').round(3)
archetype_table['team_id'] = archetype_table['team_id'].astype(str).str.title()
archetype_table

### 4. Representative Shape Examples

Pick one possession for a few common shape tags and overlay them. This is a starting point for story visuals like "the direct strike," "the reset grind," or "the middle conveyor belt."

In [ ]:
representative_rows = []
for tag in ['direct', 'middle-heavy', 'sideline-heavy', 'side-switching', 'reset-heavy']:
    matches = story_possessions[story_possessions['shape_tags'].str.contains(tag, regex=False, na=False)]
    if not matches.empty:
        representative_rows.append(matches.sort_values('aec_per_throw', ascending=False).iloc[0])

representatives = pd.DataFrame(representative_rows).drop_duplicates('possession_id')
representative_paths = label_ranked_paths(representatives, story_paths, 'aec_per_throw', 'Shape')
plot_representative_paths(
    representative_paths,
    title='Representative high-aEC possession shapes',
    show_arrows=False,
)

### 5. Who Does This Consistently?

The top-five chart is a highlight reel. This section asks which teams repeatedly produce strong non-huck long-field possessions.

In [ ]:
scoring_consistency = summarize_team_aec_consistency(
    story_possessions,
    metric='aec_per_throw',
    min_possessions=CONSISTENCY_MIN_POSSESSIONS,
    min_throw_count=CONSISTENCY_MIN_THROWS,
    top_n=CONSISTENCY_TOP_N,
    sort_by='median_metric',
)

display_columns = [
    'rank', 'team_id', 'median_metric', 'p75_metric', 'top_n_mean_metric',
    'high_metric_share', 'team_aec_per_throw', 'possessions', 'throws',
]
consistency_table = scoring_consistency.reindex(columns=display_columns).copy()
for column in ['median_metric', 'p75_metric', 'top_n_mean_metric', 'high_metric_share', 'team_aec_per_throw']:
    consistency_table[column] = pd.to_numeric(consistency_table[column], errors='coerce').round(3)
consistency_table['team_id'] = consistency_table['team_id'].astype(str).str.title()
consistency_table.head(12)

### 6. Turnover-Inclusive Reality Check

A scoring-only possession-shape story can make weak teams look interesting because it ignores failed possessions. This check includes turnovers while keeping the long-field, non-huck lens.

In [ ]:
turnover_inclusive_pool, turnover_inclusive_paths = filter_analysis_possessions(
    league_all_possessions,
    league_all_paths,
    outcomes=None,
    line_type='o_line',
    long_field_only=True,
    max_start_y=45,
    min_field_progress=0,
    exclude_hucks=True,
)

turnover_inclusive_consistency = summarize_team_aec_consistency(
    turnover_inclusive_pool,
    metric='aec_per_throw',
    min_possessions=CONSISTENCY_MIN_POSSESSIONS,
    min_throw_count=CONSISTENCY_MIN_THROWS,
    top_n=CONSISTENCY_TOP_N,
    sort_by='median_metric',
)

turnover_columns = [
    'rank', 'team_id', 'median_metric', 'p75_metric', 'team_aec_per_throw',
    'goal_share', 'turnover_share', 'possessions', 'throws',
]
turnover_table = turnover_inclusive_consistency.reindex(columns=turnover_columns).copy()
for column in ['median_metric', 'p75_metric', 'team_aec_per_throw', 'goal_share', 'turnover_share']:
    turnover_table[column] = pd.to_numeric(turnover_table[column], errors='coerce').round(3)
turnover_table['team_id'] = turnover_table['team_id'].astype(str).str.title()
turnover_table.head(12)

## Takeaways

After running the notebook, use the tables and visuals above to write concrete takeaways. Good first questions:

- Which possessions appear in both the `aec_per_throw` and `total_aec` top five?
- Do the best `aec_per_throw` possessions look more direct than the best `total_aec` possessions?
- Which teams have a strong top-five highlight reel but weaker median possession quality?
- Which shape tags feel real enough to become recurring article categories?
- How much does the story change when turnovers are included?